In [1]:
# Uninstall the conflicting versions first to be safe
!pip uninstall -y torch torchvision torchaudio

# Install the evaluation libraries, but keep the standard Colab torch versions
!pip install bert-score rouge-score sacrebleu

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl (530.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
timm 1.0.26 requires torchvision, which is not installed.


In [3]:
import pandas as pd

# Load the id and correct answers
reference_df = pd.read_csv("aut_tax_law.csv")[["id", "correct_answer"]]

# Load the 3 model outputs
model1_df = pd.read_csv("output_model1_kollarczik.csv")
model2_df = pd.read_csv("output_model2_kollarczik.csv")
model3_df = pd.read_csv("output_model3_kollarczik.csv")

# Merge each model output with the correct answers on 'id'
m1 = model1_df.merge(reference_df, on="id")
m2 = model2_df.merge(reference_df, on="id")
m3 = model3_df.merge(reference_df, on="id")

print(f"Model 1: {len(m1)} rows matched")
print(f"Model 2: {len(m2)} rows matched")
print(f"Model 3: {len(m3)} rows matched")
print("\nPreview:")
print(m1.head(2))

Model 1: 634 rows matched
Model 2: 643 rows matched
Model 3: 643 rows matched

Preview:
             id                                             answer  \
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...   
1  CORP-TAX-002  Wenn eine Körperschaft ein zinsloses Darlehen ...   

                                      correct_answer  
0  Das Einkommen, das der unbeschränkt Steuerpfli...  
1  Die nicht verrechneten Zinsen stellen eine ver...  


In [4]:
def clean_dataframe(df):
    # Fill empty values with a placeholder string
    df['answer'] = df['answer'].fillna("no_response")
    df['correct_answer'] = df['correct_answer'].fillna("no_answer")

    # Change everything to string type
    df['answer'] = df['answer'].astype(str)
    df['correct_answer'] = df['correct_answer'].astype(str)

    # Strip extra whitespace/newlines
    df['answer'] = df['answer'].str.strip()
    df['correct_answer'] = df['correct_answer'].str.strip()

    return df

# Apply to all three merged dataframes
m1 = clean_dataframe(m1)
m2 = clean_dataframe(m2)
m3 = clean_dataframe(m3)

print("Data cleaning complete.")

Data cleaning complete.


In [11]:
from bert_score import score
from rouge_score import rouge_scorer
import sacrebleu

def evaluate_model(df, model_name):
    # Prepare lists
    candidates = df['answer'].astype(str).tolist()
    references = df['correct_answer'].astype(str).tolist()

    # 1. Calculate BERTScore
    P, R, F1 = score(candidates, references, lang="de", verbose=False)
    avg_bert_f1 = F1.mean().item()

    # 2. Calculate ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(ref, cand)['rougeL'].fmeasure for ref, cand in zip(references, candidates)]
    avg_rouge_l = sum(rouge_scores) / len(rouge_scores)

    # 3. Calculate BLEU (SacreBLEU)
    # .score returns a float between 0 and 100; divide by 100 to get 0 to 1
    bleu_result = sacrebleu.corpus_bleu(candidates, [references])
    bleu_score = round(bleu_result.score / 100, 4)

    return {
        "Model": model_name,
        "BLEU": round(bleu_score, 4),
        "ROUGE-L": round(avg_rouge_l, 4),
        "BERTScore": round(avg_bert_f1, 4)
    }

# Run the evaluation for all three
results = []
results.append(evaluate_model(m1, "Inference"))
results.append(evaluate_model(m2, "Fine tuned"))
results.append(evaluate_model(m3, "RAG"))

# Create the final result table
results_df = pd.DataFrame(results)
print("\n--- Final Evaluation Table ---")
print(results_df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Final Evaluation Table ---
        Model    BLEU  ROUGE-L  BERTScore
0   Inference  0.0417   0.1849     0.7157
1  Fine tuned  0.0176   0.1067     0.6555
2         RAG  0.0197   0.0988     0.6644
